# 02 - 模型定义对比: nn.Module vs PreTrainedModel

对比 from-scratch GPT (纯 PyTorch) 与 HuggingFace ClearMindForCausalLM 的设计差异。

| 维度 | from-scratch (GPT) | HuggingFace (ClearMindForCausalLM) |
|------|-------------------|------------------------------------|
| 基类 | `nn.Module` | `PreTrainedModel` |
| 配置 | `dataclass ModelConfig` | `PretrainedConfig` 子类 |
| 保存/加载 | `torch.save/load` | `save_pretrained/from_pretrained` |
| 生成 | 手写 `generate()` | `GenerationMixin.generate()` |
| AutoClass | 无 | `AutoConfig/AutoModelForCausalLM` |

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
from model import ClearMindConfig, ClearMindForCausalLM

## 1. 配置定义方式

In [ ]:
# === HuggingFace 方式: PretrainedConfig ===
config = ClearMindConfig.tiny()
print(f"model_type: {config.model_type}")
print(f"hidden_size: {config.hidden_size}")
print(f"head_dim: {config.head_dim}")
print(f"num_key_value_groups: {config.num_key_value_groups}")
print(f"\n参数量估算: {config.count_params()['total_millions']:.2f}M")

# 对比: from-scratch 使用 dataclass
# config = ModelConfig.tiny()  # d_model=128, n_heads=4, ...
# HF 命名: d_model → hidden_size, n_heads → num_attention_heads

## 2. 模型实例化与前向传播

In [ ]:
model = ClearMindForCausalLM(config)
print(f"参数量: {model.count_parameters()['total_millions']:.2f}M")
print(f"\n模型结构:")
print(model)

In [ ]:
# === HuggingFace 风格: 返回 dataclass ===
input_ids = torch.randint(0, config.vocab_size, (2, 16))
labels = input_ids.clone()

outputs = model(input_ids, labels=labels)
print(f"返回类型: {type(outputs).__name__}")
print(f"logits: {outputs.logits.shape}")
print(f"loss: {outputs.loss.item():.4f}")
print(f"past_key_values: {type(outputs.past_key_values)}")

# 对比: from-scratch 返回 tuple
# logits, loss, kv_caches = model(input_ids, targets)

## 3. Save / Load

In [ ]:
import tempfile

# === HuggingFace: save_pretrained / from_pretrained ===
with tempfile.TemporaryDirectory() as tmpdir:
    model.save_pretrained(tmpdir)
    loaded = ClearMindForCausalLM.from_pretrained(tmpdir)
    
    # 验证输出一致
    with torch.no_grad():
        orig_out = model(input_ids).logits
        load_out = loaded(input_ids).logits
    print(f"Save/Load 一致: {torch.allclose(orig_out, load_out, atol=1e-5)}")

# 对比: from-scratch 使用 torch.save/load
# torch.save(model.state_dict(), 'model.pt')
# model.load_state_dict(torch.load('model.pt'))

## 4. Generate 接口

In [ ]:
# === HuggingFace: model.generate() ===
model.eval()
prompt = torch.randint(0, config.vocab_size, (1, 4))

with torch.no_grad():
    output = model.generate(
        prompt,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        top_k=50,
    )

print(f"Prompt: {prompt.shape} → Generated: {output.shape}")
print(f"新生成 tokens: {output.shape[1] - prompt.shape[1]}")

# 对比: from-scratch 需要手写 generate 函数
# from inference.generate import generate
# output = generate(model, prompt, max_new_tokens=20, temperature=0.8)

## 5. AutoClass 注册

In [ ]:
# === HuggingFace: AutoClass 自动发现 ===
from model.auto_register import *  # 触发注册
from transformers import AutoConfig, AutoModelForCausalLM

auto_config = AutoConfig.for_model("clearmind", hidden_size=64, num_hidden_layers=2, vocab_size=500)
auto_model = AutoModelForCausalLM.from_config(auto_config)

print(f"AutoConfig 类型: {type(auto_config).__name__}")
print(f"AutoModel 类型: {type(auto_model).__name__}")
print(f"参数量: {auto_model.count_parameters()['total_millions']:.3f}M")

# from-scratch 版没有 AutoClass 支持

## 6. Weight Tying 验证

In [ ]:
# 验证 embed_tokens 和 lm_head 共享同一块内存
embed_ptr = model.model.embed_tokens.weight.data_ptr()
lm_head_ptr = model.lm_head.weight.data_ptr()
print(f"embed_tokens 内存地址: {embed_ptr}")
print(f"lm_head 内存地址:     {lm_head_ptr}")
print(f"Weight Tying: {embed_ptr == lm_head_ptr}")

## 总结

| 功能 | from-scratch | HuggingFace |
|------|-------------|-------------|
| 配置 | `dataclass` 手写 | `PretrainedConfig` 继承 |
| 前向传播 | 返回 `tuple` | 返回 `CausalLMOutputWithPast` |
| 保存加载 | `torch.save/load` | `save_pretrained/from_pretrained` |
| 生成 | 手写循环 | `GenerationMixin.generate()` |
| 注册 | 无 | `AutoConfig/AutoModel` |
| 生态 | 独立 | pipeline, Trainer, PEFT, TRL... |

**核心收获:** 模型架构完全相同 (RoPE + RMSNorm + SwiGLU + GQA + KV Cache)，
区别在于接口适配 — HuggingFace 提供标准化接口，从而融入整个生态系统。